# 실습 12: 한 번 잰 점수를 믿어도 되나
- 상황: 재현율을 올렸는데, 그 점수는 한 번 나눠서 잰 것이다
- 목표: 여러 번 나눠 재고, 평균과 흔들림을 함께 적는다

## Step 0. 앞 실습까지 재현하기

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("../../day02/lab06_clean-dataset/results/secom_clean.csv")

sensor_cols = df.columns.drop("result")
df[sensor_cols] = df[sensor_cols].fillna(df[sensor_cols].median())

df["불량여부"] = (df["result"] == "불량").astype(int)

X = df[sensor_cols]
y = df["불량여부"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("학습용:", X_train.shape[0], "건, 불량", int(y_train.sum()), "건")
print("시험용:", X_test.shape[0], "건, 불량", int(y_test.sum()), "건")

학습용: 1253 건, 불량 83 건
시험용: 314 건, 불량 21 건


## Step 1. 오늘 쓸 말 정리하기

### 용어 풀이 - 여러 번 재는 말

| 말 | 뜻 |
|---|---|
| 교차검증 | 학습용을 여러 덩어리로 나누고, 돌아가며 한 덩어리씩 시험지로 써서 여러 번 재는 방법 |
| 겹 | 나눈 덩어리 하나. 다섯 겹이면 다섯 번 재게 된다 |
| 평균 | 여러 번 잰 점수의 가운데 값 |
| 흔들림 (표준편차) | 잰 값들이 평균에서 얼마나 벌어져 있나. 클수록 들쭉날쭉하다는 뜻 |
| 층화 | 겹마다 드문 쪽 비율을 원래대로 맞춰 나누는 것. 나눌 때 썼던 그 개념 |

## Step 2. 나누는 방식만 바꿔 다섯 번 재보기

In [2]:
# 필요한 도구들을 불러온다
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import recall_score, f1_score

# random_state - 무작위로 섞는 방식을 고정하는 번호. 번호가 다르면 다르게 섞인다
for 번호 in [1, 42, 7, 100, 2024]:
    # 같은 데이터를 번호만 바꿔 다시 나눈다
    학습입력, 시험입력, 학습정답, 시험정답 = train_test_split(
        X, y, test_size=0.2, random_state=번호, stratify=y)

    # 모델도 설정도 완전히 같다. 바뀐 것은 나눈 방식뿐이다
    모델 = make_pipeline(StandardScaler(),
                       LogisticRegression(max_iter=1000, class_weight="balanced"))
    모델.fit(학습입력, 학습정답)
    예측 = 모델.predict(시험입력)

    print(f"번호 {번호}: 정확도 {round((예측 == 시험정답).mean() * 100, 2)}%",
          f"재현율 {round(recall_score(시험정답, 예측), 3)}",
          f"F1 {round(f1_score(시험정답, 예측), 3)}")

번호 1: 정확도 77.71% 재현율 0.619 F1 0.271
번호 42: 정확도 75.16% 재현율 0.429 F1 0.188
번호 7: 정확도 73.89% 재현율 0.762 F1 0.281
번호 100: 정확도 79.3% 재현율 0.619 F1 0.286
번호 2024: 정확도 73.25% 재현율 0.571 F1 0.222


## Step 3. 교차검증으로 한 번에 재기

In [3]:
# 교차검증 도구를 불러온다
from sklearn.model_selection import cross_val_score, StratifiedKFold
import numpy as np

# StratifiedKFold - 겹마다 불량 비율을 원래대로 맞춰가며 다섯 덩어리로 나눈다
겹나누기 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

모델 = make_pipeline(StandardScaler(),
                   LogisticRegression(max_iter=1000, class_weight="balanced"))

# 학습용 안에서만 다섯 번 재고 그 점수 다섯 개를 돌려준다. 시험용은 넣지 않는다
점수들 = cross_val_score(모델, X_train, y_train, cv=겹나누기, scoring="recall")

# float(v) - 넘파이 숫자를 파이썬 숫자로 바꾼다. 안 바꾸면 np.float64(0.529) 처럼 이름표가 붙어 나온다
print("겹마다의 재현율:", [round(float(v), 3) for v in 점수들])
print("평균:", round(점수들.mean(), 3))
print("흔들림:", round(점수들.std(), 3))

겹마다의 재현율: [0.529, 0.706, 0.647, 0.562, 0.812]
평균: 0.651
흔들림: 0.102


### 문법 노트 - 여러 번 재기

| 쓴 것 | 하는 일 | 왜 여기 쓰나 |
|---|---|---|
| StratifiedKFold(n_splits=5) | 다섯 덩어리로 나누되 겹마다 불량 비율을 맞춘다 | 그냥 나누면 어떤 겹에 불량이 거의 없을 수 있다 |
| cross_val_score(..., cv=..) | 겹마다 학습하고 채점해서 점수를 모아준다 | 다섯 번 따로 쓸 코드를 한 줄로 줄인다 |
| scoring="recall" | 무엇을 잴지 정한다 | 안 정하면 정확도로 잰다. 우리 문제에선 그게 함정 |
| .std() | 흔들림(표준편차)을 구한다 | 평균만 보면 들쭉날쭉한 걸 놓친다 |
| float(값) | 넘파이 숫자를 평범한 소수로 바꾼다 | 안 바꾸면 목록 안에서 np.float64(0.529)처럼 이름표가 붙어 나온다 |

---
## Step 4. 세 모델을 같은 방식으로 재기

In [4]:
# 나무 모델도 불러온다
from sklearn.tree import DecisionTreeClassifier

# 세 모델 다 같은 겹나누기(StratifiedKFold, 5겹, shuffle=True, random_state=42)로 재고,
# 학습용(X_train, y_train)만 넣는다 - 시험용은 여기서 전혀 쓰지 않는다
모델들 = {
    "로지스틱 (손 안 댐)": make_pipeline(StandardScaler(), LogisticRegression()),
    "로지스틱 (가중치)": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight="balanced")),
    "나무 (깊이10, 끝자리10)": DecisionTreeClassifier(class_weight="balanced", random_state=42, max_depth=10, min_samples_leaf=10),
}

결과 = []
for 이름, 모델 in 모델들.items():
    재현율들 = cross_val_score(모델, X_train, y_train, cv=겹나누기, scoring="recall")
    F1들 = cross_val_score(모델, X_train, y_train, cv=겹나누기, scoring="f1")

    print(f"[{이름}]")
    print("  겹마다의 재현율:", [round(float(v), 3) for v in 재현율들])
    print("  겹마다의 F1    :", [round(float(v), 3) for v in F1들])

    결과.append({
        "모델": 이름,
        "재현율 평균": round(재현율들.mean(), 3),
        "재현율 흔들림": round(재현율들.std(), 3),
        "F1 평균": round(F1들.mean(), 3),
        "F1 흔들림": round(F1들.std(), 3),
    })

print()
print(f"학습용 {len(y_train)} 건 (불량 {int(y_train.sum())} 건) 안에서만 다섯 번씩 재고 평균낸 값")
print(f"시험용 {len(y_test)} 건은 이 칸에 한 번도 들어가지 않았다")
print()

교차검증_비교표 = pd.DataFrame(결과).set_index("모델")
교차검증_비교표

[로지스틱 (손 안 댐)]
  겹마다의 재현율: [0.059, 0.176, 0.118, 0.0, 0.188]
  겹마다의 F1    : [0.105, 0.25, 0.182, 0.0, 0.273]


[로지스틱 (가중치)]
  겹마다의 재현율: [0.529, 0.706, 0.647, 0.562, 0.812]
  겹마다의 F1    : [0.261, 0.25, 0.275, 0.234, 0.252]


[나무 (깊이10, 끝자리10)]
  겹마다의 재현율: [0.294, 0.412, 0.471, 0.438, 0.25]
  겹마다의 F1    : [0.118, 0.173, 0.19, 0.179, 0.125]

학습용 1253 건 (불량 83 건) 안에서만 다섯 번씩 재고 평균낸 값
시험용 314 건은 이 칸에 한 번도 들어가지 않았다



,재현율 평균,재현율 흔들림,F1 평균,F1 흔들림
모델,,,,
로지스틱 (손 안 댐),0.108,0.071,0.162,0.100
로지스틱 (가중치),0.651,0.102,0.254,0.014
"나무 (깊이10, 끝자리10)",0.373,0.085,0.157,0.030


In [ ]:
# Step 5 표의 "시험용 재현율" 칸에 넣을 값
# 학습용 전체(X_train, y_train)로 딱 한 번 학습시키고, 시험용(X_test)으로 딱 한 번 재본다
# 교차검증(cross_val_score)은 여기서 만든 모델 객체를 복제해서 썼을 뿐, 원본은 아직 학습되지 않은 상태였다
시험용_결과 = []
for 이름, 모델 in 모델들.items():
    모델.fit(X_train, y_train)
    시험예측 = 모델.predict(X_test)
    시험용_결과.append({
        "모델": 이름,
        "시험용 재현율": round(recall_score(y_test, 시험예측), 3),
    })

pd.DataFrame(시험용_결과).set_index("모델")

## Step 5. 모델 비교표

| 모델 | 처리 | 설정 | 재현율 (평균 ± 흔들림) | F1 (평균 ± 흔들림) | 시험용 재현율 |
|---|---|---|---|---|---|
| 로지스틱 회귀 | 손 안 댐 | 기본값 | [0.108] ± [0.071] | [0.162] ± [0.100] | [0.095] |
| 로지스틱 회귀 | 가중치 | 기본값 | [0.651] ± [0.102] | [0.254] ± [0.014] | [0.429] |
| 의사결정나무 | 가중치 | [max_depth=10, min_samples_leaf=10] | [0.373] ± [0.085] | [0.157] ± [0.030] | [0.381] |

## Step 6. 오늘 택한 것

- 택한 모델 : [로지스틱 회귀 + 가중치 (class_weight="balanced")]
- 왜 : [교차검증 재현율 평균이 세 모델 중 가장 높았다(0.651, 나무 0.373·손 안 댐 0.108 대비 압도적). F1 흔들림도 세 모델 중 가장 작아(0.014) 다섯 겹에 걸쳐 비교적 안정적으로 나왔고, 시험용 단일 채점에서도 재현율 0.429로 나무(0.381)를 앞섰다]
- 무엇을 내줬나 : [시험용에서 불량이라 예측한 75건 중 진짜 불량은 9건뿐이었다 — 헛경보 66건, 정밀도 0.12까지 떨어졌다. 불량 하나를 잡으려면 양품 약 7개를 함께 세워야 하는 셈이다]
- 아직 못 미더운 점 : [교차검증 재현율이 겹마다 0.529~0.812로 크게 벌어진다(흔들림 0.102) — 어느 겹으로 재느냐에 따라 결과가 꽤 달라진다는 뜻이다. 게다가 시험용 단일 재현율(0.429)이 교차검증 평균(0.651)보다 상당히 낮게 나와, 다음에 새 데이터가 들어왔을 때 0.65를 기대하기보다는 0.4~0.5대를 더 현실적인 기준선으로 잡는 게 안전해 보인다]